# Build MIMIC-III dataset with hourly data
The raw dataset was built with MIMIC_Extract package: <a href="https://github.com/MLforHealth/MIMIC_Extract" target="_blank">link</a>

# Import packages

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1.Load data
The output of MIMIC_Extract is a .hdf file and contains four tables:
* patients: static demographics, static outcomes
* vitals_labs: time-varying vitals and labs (hourly mean, count and standard deviation)
* vitals_labs_mean: time-varying vitals and labs (hourly mean only)
* interventions: hourly binary indicators for administered interventions

In [2]:
dir = "../data/mimic/"

hdf_file_path = dir + "all_hourly_data.hdf"

df_patients = pd.read_hdf(hdf_file_path, key="patients")
df_vitals_labs = pd.read_hdf(hdf_file_path, key="vitals_labs_mean")
# df_vitals_labs = pd.read_hdf(hdf_file_path, key="vitals_labs")
# df_interventions = pd.read_hdf(hdf_file_path, key="interventions")

# 2.Data processing

## 2.1 Patient
* Extract patients of age > 18 and < 89
* No future readmission
* Select a subset of static features
* Regroup ethnicity

In [3]:
df_patients_proc = df_patients[
    (df_patients["age"] > 18)
    & (df_patients["age"] < 89)
    & (df_patients["readmission_30"] == 0)
]

In [4]:
# Select a subset of static variables

selected_vars = [
    "gender",
    "age",
    "ethnicity",
    "admission_type",
    "los_icu",
    "mort_hosp",
]

df_patients_proc = df_patients_proc[selected_vars]

In [5]:
# Regroup ethnicity

df_patients_proc["ethnicity"] = df_patients_proc["ethnicity"].str.split("/").str[0]
df_patients_proc["ethnicity"] = df_patients_proc["ethnicity"].str.split(" - ").str[0]
df_patients_proc["ethnicity"] = df_patients_proc["ethnicity"].replace(
    [
        "UNKNOWN",
        "PATIENT DECLINED TO ANSWER",
        "MULTI RACE ETHNICITY",
        "AMERICAN INDIAN",
        "UNABLE TO OBTAIN",
        "PORTUGUESE",
        "CARIBBEAN ISLAND",
        "SOUTH AMERICAN",
        "NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER",
        "MIDDLE EASTERN",
    ],
    "OTHER",
)
df_patients_proc["ethnicity"] = df_patients_proc["ethnicity"].replace(
    ["HISPANIC OR LATINO"], "HISPANIC"
)

In [6]:
df_patients_proc.describe(include="all")

,gender,age,ethnicity,admission_type,los_icu,mort_hosp
count,31135,31135.000000,31135,31135,31135.000000,31135.000000
unique,2,NaN,5,3,NaN,NaN
top,M,NaN,WHITE,EMERGENCY,NaN,NaN
freq,17980,NaN,22021,24821,NaN,NaN
mean,NaN,62.348948,NaN,NaN,2.623475,0.088967
std,NaN,16.822707,NaN,NaN,1.975600,0.284701
min,NaN,18.022085,NaN,NaN,0.500000,0.000000
25%,NaN,51.567451,NaN,NaN,1.163432,0.000000
50%,NaN,64.296748,NaN,NaN,1.969213,0.000000
75%,NaN,76.129561,NaN,NaN,3.281829,0.000000


In [7]:
print(f"{len(set(df_patients_proc.index.levels[0]))} patients")
print(f"{len(set(df_patients_proc.index.levels[1]))} hospital admins")
print(f"{len(set(df_patients_proc.index.levels[2]))} icu stays")

34472 patients
34472 hospital admins
34472 icu stays


## 2.2 Vital signs and labs
* Select a subset of variables
* Extract first and last measures within the defined window and for the selected cohort
* Remove NAs

In [8]:
# Specify a window size in hours
window_size = 48

vs_labs_vars = [
    "diastolic blood pressure",
    "fraction inspired oxygen",
    "glascow coma scale total",
    "glucose",
    "heart rate",
    "height",
    "mean blood pressure",
    "oxygen saturation",
    "systolic blood pressure",
    "temperature",
    "weight",
    "ph",
]

index = ["subject_id", "hadm_id", "icustay_id"]

In [9]:
df_vitals_labs_proc = df_vitals_labs[vs_labs_vars]

df_vitals_labs_proc = df_vitals_labs_proc[
    (
        df_vitals_labs_proc.index.get_level_values("icustay_id").isin(
            set(df_patients_proc.index.get_level_values("icustay_id"))
        )
    )
    & (df_vitals_labs_proc.index.get_level_values("hours_in") <= window_size)
]

df_vitals_labs_proc.columns = df_vitals_labs_proc.columns.droplevel(
    level=["Aggregation Function"]
)
df_vitals_labs_proc.columns.name = None

In [10]:
df_vitals_labs_proc_first = (
    df_vitals_labs_proc.groupby(index).first().add_suffix("_first")
)
df_vitals_labs_proc_last = df_vitals_labs_proc.groupby(index).last().add_suffix("_last")

df_vitals_labs_proc_first_last = df_vitals_labs_proc_first.merge(
    df_vitals_labs_proc_last, left_index=True, right_index=True
)

In [11]:
# Remove the entire column if 40% of the data is missing

df_count = df_vitals_labs_proc_first_last.describe().loc["count", :]
col_to_remove = df_count[
    df_count <= df_vitals_labs_proc_first_last.shape[0] * 0.6
].index.to_list()

df_vitals_labs_proc_first_last = df_vitals_labs_proc_first_last.drop(
    col_to_remove, axis=1
)

In [12]:
# Remove rows that have missing value

df_vitals_labs_proc_first_last = df_vitals_labs_proc_first_last.dropna()

In [13]:
df_vitals_labs_proc_first_last.shape

(15118, 18)

# 3.Save data
* Join the dataframes
* Split into train and test sets
* Save to files

In [14]:
# Combine static and time varying variables

df_combined = df_patients_proc.merge(
    df_vitals_labs_proc_first_last, left_index=True, right_index=True
)

In [15]:
# Drop ids

df_combined = df_combined.reset_index(drop=True)
df_combined.shape

(15118, 24)

In [16]:
# Split into train and test sets

df_train, df_test = train_test_split(
    df_combined, test_size=0.3, random_state=42, stratify=df_combined["mort_hosp"]
)

In [17]:
# Save data

df_combined.to_csv(dir + "mimic_extract.csv", index=False)
df_train.to_csv(dir + "mimic_extract_train.csv", index=False)
df_test.to_csv(dir + "mimic_extract_test.csv", index=False)